# EE 371 FM Demodulation Project (Mono L+R)

This notebook:
1) Plots the amplitude spectrum of the received FM signal  
2) FM-demodulates to instantaneous frequency and plots its spectrum  
3) Low-pass filters to isolate the **L+R (mono)** component  
4) Extracts the message signal m(t) and plots its spectrum  
5) Applies a de-emphasis filter and plots its spectrum  
6) Reconstructs audio at 48 kHz, plots its spectrum, and saves a WAV file


In [ ]:
import numpy as np
import scipy.io as sio
import scipy.signal as signal
from scipy.io import wavfile
import matplotlib.pyplot as plt

# Make plots consistent and readable
plt.rcParams.update({
    "figure.figsize": (6, 6),   # square figures (project requirement)
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10
})

In [ ]:
data = sio.loadmat("fm_signal.mat")
x = data["band_pass_signal"].flatten().astype(np.float64)

fs = 240_000          # Hz (given)
faudio = 48_000       # Hz (given)
decim = fs // faudio  # should be 5

assert fs % faudio == 0, "fs must be an integer multiple of faudio for simple decimation."

t = np.arange(len(x)) / fs
print(f"N = {len(x)} samples, duration = {len(x)/fs:.2f} s, fs = {fs/1e3:.0f} kHz")


In [ ]:
def amp_spectrum(sig, fs_hz, *, title, fmax_hz=None, remove_dc=True, ax=None, label=None):
    """
    Two-sided (normal) amplitude spectrum using FFT + Hann window.
    - Frequency axis in kHz, centered at 0 (fftshift)
    - Amplitude in "signal units" (relative), consistent across plots
    """
    sig = np.asarray(sig)
    if remove_dc:
        sig = sig - np.mean(sig)

    N = len(sig)
    w = np.hanning(N)

    X = np.fft.fft(sig * w)
    f = np.fft.fftfreq(N, d=1/fs_hz)

    # center zero frequency
    X = np.fft.fftshift(X)
    f = np.fft.fftshift(f)

    # Window amplitude compensation (same idea as before)
    Xmag = np.abs(X) / (np.sum(w)/2)

    if ax is None:
        fig, ax = plt.subplots()

    ax.plot(f/1e3, Xmag, linewidth=1.0, label=label)
    ax.set_title(title)
    ax.set_xlabel("Frequency (kHz)")
    ax.set_ylabel("Amplitude")

    if fmax_hz is not None:
        ax.set_xlim(-fmax_hz/1e3, fmax_hz/1e3)

    if label is not None:
        ax.legend()

    return f, Xmag


In [ ]:
fig, ax = plt.subplots()
amp_spectrum(x, fs, title="Received FM Signal: Amplitude Spectrum", fmax_hz=120_000, ax=ax)
plt.tight_layout()
plt.show()


**Figure 1.** Amplitude spectrum of the received FM signal sampled at 240 kHz.  
This plot is used to confirm the occupied bandwidth and locate the channel content before demodulation.


In [ ]:
# Analytic signal + phase derivative FM demod
xa = signal.hilbert(x)    # analytic signal
phi = np.unwrap(np.angle(xa))    # unwrapped phase (rad)
dphi = np.diff(phi)    # phase difference per sample

# Instantaneous frequency (Hz): f_inst[n] = (fs / (2π)) * Δφ[n]
f_inst = (fs / (2*np.pi)) * dphi

# Remove DC (center frequency term) to focus on message components
f_inst = f_inst - np.mean(f_inst)

fig, ax = plt.subplots(figsize=(7, 7))
amp_spectrum(f_inst, fs, title="Instantaneous Frequency (FM Demod Output): Spectrum",
             fmax_hz=120_000, ax=ax, label="Demodulated Signal")

# Highlight key regions
ax.axvspan(-15, 15, alpha=0.15, color='green', label="Mono L+R Band (±0–15 kHz)")
ax.axvspan(-53, -23, alpha=0.10, color='blue')
ax.axvspan(23, 53, alpha=0.10, color='blue', label="Stereo Region (±23–53 kHz)")

ax.set_title("Spectrum of Instantaneous Frequency\n(FM Demodulator Output)", fontsize=14, fontweight='bold')
ax.set_xlabel("Frequency (kHz)", fontsize=12)
ax.set_ylabel("Amplitude", fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(-120, 120)
plt.tight_layout()
plt.show()

**Figure 2.** Spectrum of the instantaneous frequency (FM demodulated signal).  
Prominent components here correspond to the broadcast baseband content. The low-frequency region contains the mono **L+R** audio band; additional peaks/regions may appear at higher audio frequencies depending on the broadcast multiplex structure.


In [ ]:
# Mono L+R audio is typically limited to about 15 kHz
f_cut = 15_000  # Hz

# FIR LPF design
numtaps = 401
h_lpf = signal.firwin(numtaps, cutoff=f_cut, fs=fs, window="hamming")

# Apply with zero-phase filtering
mono_baseband = signal.filtfilt(h_lpf, [1.0], f_inst)

fig, ax = plt.subplots(figsize=(7, 7))
amp_spectrum(mono_baseband, fs, title="After LPF (Isolated L+R Band): Spectrum",
             fmax_hz=30_000, ax=ax, label="Filtered Signal")

# Shade the passband
#ax.axvspan(0, 15, alpha=0.2, color='green', label="Passband (0–15 kHz)")
ax.axvline(15, color='red', linestyle='--', alpha=0.8, linewidth=1.5, label="Cutoff = 15 kHz")
ax.axvline(-15, color='red', linestyle='--', alpha=0.8, linewidth=1.5)

ax.set_title("Low-Pass Filtered Signal", fontsize=14, fontweight='bold')
ax.set_xlabel("Frequency (kHz)", fontsize=12)
ax.set_ylabel("Amplitude", fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(-30, 30)
plt.tight_layout()
plt.show()

**Figure 3.** Spectrum after low-pass filtering the demodulated signal to isolate the **L+R (mono)** band (0–15 kHz).  
This step removes higher-frequency multiplex components so only the mono audio remains.


In [ ]:
# Downsample from 240 kHz to 48 kHz (factor 5)
m = signal.resample_poly(mono_baseband, up=1, down=decim)
m = m - np.mean(m)  # remove any residual DC

fig, ax = plt.subplots()
amp_spectrum(m, faudio, title="Extracted Message m(t) at 48 kHz", fmax_hz=24_000, ax=ax)
plt.tight_layout()
plt.show()


**Figure 4.** Spectrum of the extracted message signal **m(t)** after downsampling to 48 kHz.  
This is the mono audio signal prior to de-emphasis.


In [ ]:
# De-emphasis time constant: commonly 75 µs (US) or 50 µs (EU).
# If audio sounds too dull/bright, try switching tau to 50e-6.
tau = 75e-6

# Analog de-emphasis: H(s) = 1 / (1 + s*tau)
b_z, a_z = signal.bilinear([1.0], [tau, 1.0], fs=faudio)
m_de = signal.lfilter(b_z, a_z, m)

fig, ax = plt.subplots()
amp_spectrum(m_de, faudio, title=f"After De-emphasis (τ = {tau*1e6:.0f} µs) Spectrum", fmax_hz=24_000, ax=ax)
plt.tight_layout()
plt.show()


**Figure 5.** Spectrum after applying the de-emphasis filter.  
De-emphasis restores the intended audio spectral balance by attenuating the higher-frequency boost introduced during FM pre-emphasis.


In [ ]:
# Normalize to avoid clipping; convert to int16 for WAV
audio = m_de - np.mean(m_de)
audio = audio / (np.max(np.abs(audio)) + 1e-12)
audio_int16 = np.int16(np.clip(audio, -1, 1) * 32767)

fig, ax = plt.subplots(figsize=(7, 7))
amp_spectrum(audio, faudio, title="Final Audio (Normalized): Spectrum",
             fmax_hz=24_000, ax=ax, label="Final Audio")

# Highlight full audio range
#ax.axvspan(0, 20, alpha=0.1, color='green', label="Audible Range (≈20 kHz)")
#ax.axvline(20, color='darkgreen', linestyle='--', alpha=0.8, linewidth=1.5)

ax.set_title("Final Reconstructed Audio Signal\n(Normalized, 48 kHz, Mono)", 
             fontsize=14, fontweight='bold')
ax.set_xlabel("Frequency (kHz)", fontsize=12)
ax.set_ylabel("Amplitude", fontsize=12)
ax.grid(True, alpha=0.3)
#ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(-24, 24)
plt.tight_layout()
plt.show()

out_name = "recovered_mono75.wav"
wavfile.write(out_name, faudio, audio_int16)
print(f"✅ Saved: {out_name} (fs = {faudio} Hz)")

from IPython.display import Audio, display
print("\n🎧 Playing recovered audio...")
display(Audio(audio, rate=faudio))

**Figure 6.** Spectrum of the final reconstructed audio (48 kHz), after de-emphasis and normalization.  
This is the signal saved to WAV and should match the recovered mono broadcast audio.
